# Installing Dependencies

In [69]:
# !pip install adapters
# !pip install datasets scikit-learn pandas

# Importing Libraries

In [102]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

import adapters
from adapters import AutoAdapterModel, AdapterConfig, AdapterTrainer
from adapters.composition import Stack

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

In [71]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: Tesla T4


# Configuration

In [72]:
BASE_MODEL = 'indolem/indobert-base-uncased'

SOURCE_DATA_PATH = 'source_balanced.csv'
TARGET_DATA_PATH = 'target_final.csv'
TEXT_COL_SOURCE = 'content'
TEXT_COL_TARGET = 'reviewContent'
LABEL_COL = 'label'

LABEL2ID = {'negative': 0, 'positive': 1}
ID2LABEL = {0: 'negative', 1: 'positive'}

MAX_LENGTH = 128
SEED = 42

DOMAIN_ADAPTER_NAME = 'domain_adapter'
DOMAIN_ADAPTER_REDUCTION = 16
DOMAIN_EPOCHS = 5
DOMAIN_BATCH_SIZE = 16
DOMAIN_LR = 1e-4
MMD_KERNEL_MUL = 2.0
MMD_KERNEL_NUM = 5

TASK_ADAPTER_NAME = 'task_adapter'
TASK_ADAPTER_REDUCTION = 16
TASK_EPOCHS = 5
TASK_BATCH_SIZE = 16
TASK_LR = 1e-4
TEST_SIZE = 0.2

OUTPUT_DIR = './indobert_udapter_ts_dt'

In [73]:
print('Configuration loaded.')
print(f'  Base model               : {BASE_MODEL}')
print(f'  Domain adapter reduction : {DOMAIN_ADAPTER_REDUCTION}')
print(f'  Task adapter reduction   : {TASK_ADAPTER_REDUCTION}')

Configuration loaded.
  Base model               : indolem/indobert-base-uncased
  Domain adapter reduction : 16
  Task adapter reduction   : 16


# Load Data

In [74]:
source_df = pd.read_csv(SOURCE_DATA_PATH)
source_df = source_df.dropna(subset=[TEXT_COL_SOURCE, LABEL_COL])
source_df[TEXT_COL_SOURCE] = source_df[TEXT_COL_SOURCE].astype(str).str.strip()
source_df = source_df[source_df[TEXT_COL_SOURCE] != '']
source_df['label_id'] = source_df[LABEL_COL].map(LABEL2ID)

print(f'Source dataset: {source_df.shape}')
print(source_df['label'].value_counts())

Source dataset: (59987, 4)
label
positive    29996
negative    29991
Name: count, dtype: int64


In [75]:
target_df = pd.read_csv(TARGET_DATA_PATH)
target_df = target_df.dropna(subset=[TEXT_COL_TARGET])
target_df[TEXT_COL_TARGET] = target_df[TEXT_COL_TARGET].astype(str).str.strip()
target_df = target_df[target_df[TEXT_COL_TARGET] != '']
target_texts = target_df[TEXT_COL_TARGET].tolist()

print(f'\nTarget dataset (unlabeled): {len(target_texts)} samples')
print(f'\nSample source : {source_df[TEXT_COL_SOURCE].iloc[0]}')
print(f'Sample target : {target_texts[0]}')


Target dataset (unlabeled): 38341 samples

Sample source : Pengiriman cepat banget
Sample target : bagus  mantap deh sesui pesanan


# Training Setup

## Load Tokenizer and Base Model

In [76]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model     = AutoAdapterModel.from_pretrained(BASE_MODEL)

adapters.init(model)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded  : {model.__class__.__name__}')
print(f'Total params  : {total_params:,}')


Model loaded  : BertAdapterModel
Total params  : 111,182,259


## MMD Loss Function

In [77]:
def gaussian_kernel(source, target, kernel_mul=2.0, kernel_num=5, fix_sigma=None):
    n_samples = source.size(0) + target.size(0)
    total     = torch.cat([source, target], dim=0)

    total0 = total.unsqueeze(0).expand(n_samples, n_samples, -1)
    total1 = total.unsqueeze(1).expand(n_samples, n_samples, -1)
    L2_distance = ((total0 - total1) ** 2).sum(2)

    if fix_sigma:
        bandwidth = fix_sigma
    else:
        bandwidth = torch.sum(L2_distance.detach()) / (n_samples ** 2 - n_samples)

    bandwidth /= kernel_mul ** (kernel_num // 2)
    bandwidth_list = [bandwidth * (kernel_mul ** i) for i in range(kernel_num)]

    kernel_val = [torch.exp(-L2_distance / bw) for bw in bandwidth_list]
    return sum(kernel_val)

In [78]:
def mmd_loss(source, target, kernel_mul=2.0, kernel_num=5, fix_sigma=None):
    batch_size = source.size(0)
    kernels    = gaussian_kernel(source, target, kernel_mul, kernel_num, fix_sigma)

    XX = kernels[:batch_size, :batch_size].mean()
    YY = kernels[batch_size:, batch_size:].mean()
    XY = kernels[:batch_size, batch_size:].mean()

    return XX + YY - 2 * XY

In [79]:
print('MMD loss defined.')
print(f'  Kernel multiplier : {MMD_KERNEL_MUL}')
print(f'  Number of kernels : {MMD_KERNEL_NUM}')

MMD loss defined.
  Kernel multiplier : 2.0
  Number of kernels : 5


##  Prepare DataLoaders for Domain Adapter

In [80]:
def tokenize_texts(texts, max_length=MAX_LENGTH):
    ds = Dataset.from_dict({'text': texts})
    ds = ds.map(
        lambda ex: tokenizer(
            ex['text'],
            truncation=True,
            max_length=max_length,
            padding=False
        ),
        batched=True,
        remove_columns=['text'],
        desc='Tokenizing'
    )
    ds.set_format(type='torch', columns=['input_ids', 'attention_mask'])
    return ds

In [81]:
source_texts_domain = source_df[TEXT_COL_SOURCE].tolist()

source_domain_ds = tokenize_texts(source_texts_domain)
target_domain_ds = tokenize_texts(target_texts)

Tokenizing:   0%|          | 0/59987 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/38341 [00:00<?, ? examples/s]

In [82]:
collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors='pt')

source_domain_loader = DataLoader(
    source_domain_ds, batch_size=DOMAIN_BATCH_SIZE,
    shuffle=True, collate_fn=collator, drop_last=True
)
target_domain_loader = DataLoader(
    target_domain_ds, batch_size=DOMAIN_BATCH_SIZE,
    shuffle=True, collate_fn=collator, drop_last=True
)

In [83]:
print(f'Source domain batches : {len(source_domain_loader)}')
print(f'Target domain batches : {len(target_domain_loader)}')

Source domain batches : 3749
Target domain batches : 2396


# Add Domain Adapter to Model

In [84]:
domain_adapter_config = AdapterConfig.load(
    'pfeiffer',
    reduction_factor=DOMAIN_ADAPTER_REDUCTION
)

model.add_adapter(DOMAIN_ADAPTER_NAME, config=domain_adapter_config)
model.train_adapter(DOMAIN_ADAPTER_NAME)
model.to(device)

BertAdapterModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31923, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttentionWithAdapters(
              (query): LoRALinearTorch(
                in_features=768, out_features=768, bias=True
                (shared_parameters): ModuleDict()
                (loras): ModuleDict()
              )
              (key): LoRALinearTorch(
                in_features=768, out_features=768, bias=True
                (shared_parameters): ModuleDict()
                (loras): ModuleDict()
              )
              (value): LoRALinearTorch(
             

In [85]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Domain adapter added : {DOMAIN_ADAPTER_NAME}')
print(f'Trainable params     : {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

Domain adapter added : domain_adapter
Trainable params     : 1,486,656 / 112,076,787 (1.33%)


In [86]:
# Verifikasi semua parameter sudah di device yang sama
devices = set(p.device for p in model.parameters())
print(f'Model devices        : {devices}')  # harus cuma 1 device

Model devices        : {device(type='cuda', index=0)}


# Model Training

## 1. Train Domain Adapter (MMD)

In [87]:
optimizer_domain = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=DOMAIN_LR
)

In [88]:
model.train()

BertAdapterModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31923, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttentionWithAdapters(
              (query): LoRALinearTorch(
                in_features=768, out_features=768, bias=True
                (shared_parameters): ModuleDict()
                (loras): ModuleDict()
              )
              (key): LoRALinearTorch(
                in_features=768, out_features=768, bias=True
                (shared_parameters): ModuleDict()
                (loras): ModuleDict()
              )
              (value): LoRALinearTorch(
             

In [92]:
# Cek dulu struktur output model
with torch.no_grad():
    sample = next(iter(source_domain_loader))
    sample = {k: v.to(device) for k, v in sample.items()}
    test_out = model(
        input_ids=sample['input_ids'],
        attention_mask=sample['attention_mask'],
        output_hidden_states=True
    )
    print(f'Output type   : {type(test_out)}')
    print(f'Output keys   : {test_out.keys() if hasattr(test_out, "keys") else dir(test_out)}')
    if hasattr(test_out, 'hidden_states') and test_out.hidden_states is not None:
        print(f'Hidden states : {len(test_out.hidden_states)} layers, shape: {test_out.hidden_states[-1].shape}')

Output type   : <class 'transformers.modeling_outputs.MaskedLMOutput'>
Output keys   : odict_keys(['logits', 'hidden_states'])
Hidden states : 13 layers, shape: torch.Size([16, 78, 768])


In [94]:
domain_losses = []

In [97]:
epoch_pbar = tqdm(range(DOMAIN_EPOCHS), desc='Epochs', position=0)

Epochs:   0%|          | 0/5 [00:00<?, ?it/s]

In [98]:
for epoch in epoch_pbar:
    epoch_loss = 0.0
    n_steps    = min(len(source_domain_loader), len(target_domain_loader))
    src_iter   = iter(source_domain_loader)
    tgt_iter   = iter(target_domain_loader)

    step_pbar = tqdm(
        range(n_steps),
        desc=f'  Epoch {epoch+1}/{DOMAIN_EPOCHS}',
        position=1,
        leave=False
    )

    for step in step_pbar:
        src_batch = {k: v.to(device) for k, v in next(src_iter).items()}
        tgt_batch = {k: v.to(device) for k, v in next(tgt_iter).items()}

        model.to(device)

        src_outputs = model(
            input_ids=src_batch['input_ids'],
            attention_mask=src_batch['attention_mask'],
            output_hidden_states=True
        )
        tgt_outputs = model(
            input_ids=tgt_batch['input_ids'],
            attention_mask=tgt_batch['attention_mask'],
            output_hidden_states=True
        )

        src_cls = src_outputs.hidden_states[-1][:, 0, :]
        tgt_cls = tgt_outputs.hidden_states[-1][:, 0, :]

        loss = mmd_loss(src_cls, tgt_cls, MMD_KERNEL_MUL, MMD_KERNEL_NUM)

        optimizer_domain.zero_grad()
        loss.backward()
        optimizer_domain.step()

        epoch_loss += loss.item()
        avg_loss    = epoch_loss / (step + 1)

        # Update step progress bar
        step_pbar.set_postfix({
            'MMD Loss': f'{avg_loss:.6f}',
            'step'    : f'{step+1}/{n_steps}'
        })

    step_pbar.close()

    avg_epoch_loss = epoch_loss / n_steps
    domain_losses.append(avg_epoch_loss)

    # Update epoch progress bar
    epoch_pbar.set_postfix({
        'avg MMD Loss': f'{avg_epoch_loss:.6f}',
        'epoch'       : f'{epoch+1}/{DOMAIN_EPOCHS}'
    })

    tqdm.write(f'Epoch {epoch+1}/{DOMAIN_EPOCHS} done — avg MMD loss: {avg_epoch_loss:.6f}')
    tqdm.write('-' * 60)

  Epoch 1/5:   0%|          | 0/2396 [00:00<?, ?it/s]

Epoch 1/5 done — avg MMD loss: 0.380002
------------------------------------------------------------


  Epoch 2/5:   0%|          | 0/2396 [00:00<?, ?it/s]

Epoch 2/5 done — avg MMD loss: 0.385480
------------------------------------------------------------


  Epoch 3/5:   0%|          | 0/2396 [00:00<?, ?it/s]

Epoch 3/5 done — avg MMD loss: 0.384801
------------------------------------------------------------


  Epoch 4/5:   0%|          | 0/2396 [00:00<?, ?it/s]

Epoch 4/5 done — avg MMD loss: 0.384644
------------------------------------------------------------


  Epoch 5/5:   0%|          | 0/2396 [00:00<?, ?it/s]

Epoch 5/5 done — avg MMD loss: 0.385501
------------------------------------------------------------


In [99]:
print(f'MMD loss per epoch: {[f"{l:.6f}" for l in domain_losses]}')

MMD loss per epoch: ['0.380002', '0.385480', '0.384801', '0.384644', '0.385501']


In [100]:
domain_adapter_dir = os.path.join(OUTPUT_DIR, 'domain_adapter')
os.makedirs(domain_adapter_dir, exist_ok=True)

model.save_adapter(domain_adapter_dir, DOMAIN_ADAPTER_NAME)

print(f'Domain adapter saved to: {domain_adapter_dir}')
for f in os.listdir(domain_adapter_dir):
    fpath   = os.path.join(domain_adapter_dir, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {f:30s} ({size_kb:.1f} KB)')

Domain adapter saved to: ./indobert_udapter_ts_dt/domain_adapter
  adapter_config.json            (1.1 KB)
  pytorch_adapter.bin            (3512.0 KB)


# Prepare Labeled Source Data for Task Adapter

In [103]:
train_df, test_df = train_test_split(
    source_df,
    test_size=TEST_SIZE,
    stratify=source_df['label_id'],
    random_state=SEED
)

In [104]:
print(f'Train: {len(train_df)} | Test: {len(test_df)}')

Train: 47989 | Test: 11998


In [105]:
def make_labeled_dataset(df, text_col):
    ds = Dataset.from_dict({
        'text'  : df[text_col].tolist(),
        'labels': df['label_id'].tolist()
    })
    ds = ds.map(
        lambda ex: tokenizer(
            ex['text'],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False
        ),
        batched=True,
        remove_columns=['text'],
        desc='Tokenizing'
    )
    return ds

In [106]:
train_dataset = make_labeled_dataset(train_df, TEXT_COL_SOURCE)
test_dataset  = make_labeled_dataset(test_df,  TEXT_COL_SOURCE)

print(f'Train dataset features: {train_dataset.features}')

Tokenizing:   0%|          | 0/47989 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/11998 [00:00<?, ? examples/s]

Train dataset features: {'labels': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}


# Add Task Adapter (Stacked on Domain Adapter)

In [107]:
task_adapter_config = AdapterConfig.load(
    'pfeiffer',
    reduction_factor=TASK_ADAPTER_REDUCTION
)

model.add_adapter(TASK_ADAPTER_NAME, config=task_adapter_config)
model.add_classification_head(TASK_ADAPTER_NAME, num_labels=2, id2label=ID2LABEL)

# Stack: domain adapter (frozen) → task adapter (trainable)
model.active_adapters = Stack(DOMAIN_ADAPTER_NAME, TASK_ADAPTER_NAME)
model.train_adapter(TASK_ADAPTER_NAME)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Task adapter added   : {TASK_ADAPTER_NAME}')
print(f'Active stack         : {DOMAIN_ADAPTER_NAME} → {TASK_ADAPTER_NAME}')
print(f'Trainable params     : {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

Task adapter added   : task_adapter
Active stack         : domain_adapter → task_adapter
Trainable params     : 2,078,786 / 113,563,445 (1.83%)


# Train Task Adapter

In [108]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy    = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

In [109]:
task_training_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, 'task_adapter_checkpoints'),
    num_train_epochs=TASK_EPOCHS,
    per_device_train_batch_size=TASK_BATCH_SIZE,
    per_device_eval_batch_size=TASK_BATCH_SIZE,
    learning_rate=TASK_LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    push_to_hub=False,
    report_to='none',
    seed=SEED
)

In [110]:
task_trainer = AdapterTrainer(
    model=model,
    args=task_training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

In [111]:
task_result = task_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.377300,0.301167,0.886648,0.889225,0.886648,0.886461
2,0.333200,0.304965,0.883981,0.883999,0.883981,0.883979
3,0.280400,0.298834,0.891565,0.893954,0.891565,0.891401
4,0.284500,0.287170,0.894899,0.896836,0.894899,0.894771
5,0.259900,0.288341,0.894816,0.896001,0.894816,0.894738


In [112]:
print(f'Training loss : {task_result.training_loss:.4f}')
print(f'Training time : {task_result.metrics["train_runtime"]:.1f} s')

Training loss : 0.3217
Training time : 968.8 s


# Evaluate on Source Test Set

In [113]:
eval_results = task_trainer.evaluate()

print(f'\n  Accuracy  : {eval_results["eval_accuracy"]:.4f}')
print(f'  Precision : {eval_results["eval_precision"]:.4f}')
print(f'  Recall    : {eval_results["eval_recall"]:.4f}')
print(f'  F1 Score  : {eval_results["eval_f1"]:.4f}')
print(f'  Loss      : {eval_results["eval_loss"]:.4f}')


  Accuracy  : 0.8949
  Precision : 0.8968
  Recall    : 0.8949
  F1 Score  : 0.8948
  Loss      : 0.2872


In [114]:
predictions = task_trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print('\nDetailed Classification Report:')
print('=' * 60)
print(classification_report(
    true_labels, pred_labels,
    target_names=['negative', 'positive'],
    digits=4
))


Detailed Classification Report:
              precision    recall  f1-score   support

    negative     0.8691    0.9298    0.8984      5998
    positive     0.9246    0.8600    0.8911      6000

    accuracy                         0.8949     11998
   macro avg     0.8968    0.8949    0.8948     11998
weighted avg     0.8968    0.8949    0.8948     11998



# Evaluate on Mixed Test Set (Source + Target)

In [115]:
mixed_df = pd.read_csv('mixed_test.csv')
mixed_df = mixed_df.dropna(subset=['content', 'label'])
mixed_df['label_id'] = mixed_df['label'].map(LABEL2ID)

print(f'Mixed test set: {mixed_df.shape}')
print(mixed_df['origin'].value_counts())

Mixed test set: (35870, 5)
origin
source    28201
target     7669
Name: count, dtype: int64


In [116]:
mixed_dataset     = make_labeled_dataset(mixed_df, 'content')
mixed_predictions = task_trainer.predict(mixed_dataset)
mixed_pred_labels = np.argmax(mixed_predictions.predictions, axis=1)
mixed_true_labels = mixed_predictions.label_ids

mixed_df['predicted_label_id'] = mixed_pred_labels
mixed_df['predicted_label']    = mixed_df['predicted_label_id'].map(ID2LABEL)

overall_acc = accuracy_score(mixed_true_labels, mixed_pred_labels)
print(f'\nOVERALL ACCURACY: {overall_acc:.4f} ({overall_acc*100:.2f}%)')
print('=' * 60)

for origin in mixed_df['origin'].unique():
    mask   = mixed_df['origin'] == origin
    o_true = mixed_df.loc[mask, 'label_id'].values
    o_pred = mixed_df.loc[mask, 'predicted_label_id'].values

    o_acc = accuracy_score(o_true, o_pred)
    o_prec, o_rec, o_f1, _ = precision_recall_fscore_support(
        o_true, o_pred, average='weighted'
    )

    print(f'\n{origin.upper()} domain ({mask.sum()} samples):')
    print(f'  Accuracy  : {o_acc:.4f}')
    print(f'  Precision : {o_prec:.4f}')
    print(f'  Recall    : {o_rec:.4f}')
    print(f'  F1        : {o_f1:.4f}')
    print(classification_report(
        o_true, o_pred,
        target_names=['negative', 'positive'],
        digits=4
    ))

Tokenizing:   0%|          | 0/35870 [00:00<?, ? examples/s]


OVERALL ACCURACY: 0.8924 (89.24%)

SOURCE domain (28201 samples):
  Accuracy  : 0.8894
  Precision : 0.8953
  Recall    : 0.8894
  F1        : 0.8898
              precision    recall  f1-score   support

    negative     0.8325    0.9338    0.8802     12278
    positive     0.9437    0.8551    0.8972     15923

    accuracy                         0.8894     28201
   macro avg     0.8881    0.8944    0.8887     28201
weighted avg     0.8953    0.8894    0.8898     28201


TARGET domain (7669 samples):
  Accuracy  : 0.9036
  Precision : 0.9102
  Recall    : 0.9036
  F1        : 0.9064
              precision    recall  f1-score   support

    negative     0.6236    0.7161    0.6667      1032
    positive     0.9548    0.9328    0.9437      6637

    accuracy                         0.9036      7669
   macro avg     0.7892    0.8244    0.8052      7669
weighted avg     0.9102    0.9036    0.9064      7669



In [117]:
# Save Final Model

In [118]:
final_dir = os.path.join(OUTPUT_DIR, 'final')
os.makedirs(final_dir, exist_ok=True)

model.save_adapter(os.path.join(final_dir, 'domain_adapter'), DOMAIN_ADAPTER_NAME)
model.save_adapter(os.path.join(final_dir, 'task_adapter'),   TASK_ADAPTER_NAME)
tokenizer.save_pretrained(final_dir)

print(f'Saved to: {final_dir}')
print('\nTo load later:')
print(f'  model = AutoAdapterModel.from_pretrained("{BASE_MODEL}")')
print(f'  model.load_adapter("{final_dir}/domain_adapter")')
print(f'  model.load_adapter("{final_dir}/task_adapter")')
print(f'  model.active_adapters = Stack("domain_adapter", "task_adapter")')

Saved to: ./indobert_udapter_ts_dt/final

To load later:
  model = AutoAdapterModel.from_pretrained("indolem/indobert-base-uncased")
  model.load_adapter("./indobert_udapter_ts_dt/final/domain_adapter")
  model.load_adapter("./indobert_udapter_ts_dt/final/task_adapter")
  model.active_adapters = Stack("domain_adapter", "task_adapter")


# Quick Inference Test

In [119]:
model.eval()

BertAdapterModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31923, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttentionWithAdapters(
              (query): LoRALinearTorch(
                in_features=768, out_features=768, bias=True
                (shared_parameters): ModuleDict()
                (loras): ModuleDict()
              )
              (key): LoRALinearTorch(
                in_features=768, out_features=768, bias=True
                (shared_parameters): ModuleDict()
                (loras): ModuleDict()
              )
              (value): LoRALinearTorch(
             

In [120]:
test_samples = [
    'Aplikasinya sangat membantu dan mudah digunakan!',
    'Pengiriman cepat, barang sesuai deskripsi. Sangat puas!',
    'Sangat kecewa dengan pelayanan. Tidak profesional!',
    'Aplikasi sering error dan lambat. Tolong diperbaiki.',
]

In [121]:
for text in test_samples:
    inputs = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=MAX_LENGTH
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs      = torch.softmax(outputs.logits, dim=-1)
    pred_id    = probs.argmax().item()
    confidence = probs[0][pred_id].item()

    print(f'Text       : {text}')
    print(f'Prediction : {ID2LABEL[pred_id].upper()} (confidence: {confidence:.4f})')
    print('-' * 60)

Text       : Aplikasinya sangat membantu dan mudah digunakan!
Prediction : POSITIVE (confidence: 0.9948)
------------------------------------------------------------
Text       : Pengiriman cepat, barang sesuai deskripsi. Sangat puas!
Prediction : POSITIVE (confidence: 0.9928)
------------------------------------------------------------
Text       : Sangat kecewa dengan pelayanan. Tidak profesional!
Prediction : NEGATIVE (confidence: 0.9408)
------------------------------------------------------------
Text       : Aplikasi sering error dan lambat. Tolong diperbaiki.
Prediction : NEGATIVE (confidence: 0.8968)
------------------------------------------------------------
